# Building an AI-Driven Cyber Threat Awareness and Detection Prototype
### Academic Mini-Project: Cyber Threat Detection using the TON_IoT Network Dataset

**Author:** Cybersecurity & Data Science Analyst  
**Dataset:** TON_IoT Network Dataset (ADFA / UNSW Canberra Cyber via Kaggle)  

---
### Project Outline
- **Task 1:** Workspace & Environment Setup
- **Task 2:** Exploratory Data Analysis (EDA) & Threat Indicator Identification
- **Task 3:** Machine Learning Threat Classification (Decision Tree Classifier)
- **Task 4:** Deep Learning Threat Classification (Multi-Layer Perceptron / MLP Classifier)
- **Task 5:** Ethical Reflection, Aggressive Threat Threshold Simulation & False Positive Impact

## Task 1: Environment & Dependency Verification
Let us import required libraries and verify system environments.

In [ ]:
# Required installations (run if running on fresh Google Colab or local environment)
# !pip install kagglehub pandas numpy scikit-learn matplotlib seaborn

import os
import glob
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import kagglehub

print("Environment initialized successfully!")

## Task 2: Dataset Acquisition & Exploratory Data Analysis (EDA)
We download the TON_IoT Network Dataset using `kagglehub`, inspect its architecture, profile normal vs. malicious telemetry, and plot network feature distributions.

In [ ]:
# Download latest version of TON_IoT Network Dataset via kagglehub
dataset_dir = kagglehub.dataset_download("arnobbhowmik/ton-iot-network-dataset")
print("Path to dataset directory:", dataset_dir)

# Locate CSV files within the downloaded directory
csv_files = glob.glob(os.path.join(dataset_dir, "**", "*.csv"), recursive=True)
print(f"Discovered {len(csv_files)} CSV files:")
for f in csv_files:
    print(" -", os.path.basename(f), f"({os.path.getsize(f) / (1024*1024):.2f} MB)")

# Select the primary dataset file (e.g., Train_Test_Network.csv or first available)
selected_csv = csv_files[0]
print(f"\nLoading primary dataset file: {selected_csv}")

# Load sample or complete dataset (reading up to 100,000 records for fast reproducible execution)
# Remove nrows=100000 if training on high-RAM GPUs or full memory environments
df = pd.read_csv(selected_csv, nrows=100000, low_memory=False)
print("Dataset Shape:", df.shape)
display(df.head())

In [ ]:
# Inspect column structure, data types, and missing values
print("--- Column Names & Missing Values ---")
missing_counts = df.isnull().sum()
print(missing_counts[missing_counts > 0])

# Verify target label columns ('label' is binary 0/1; 'type' is multi-class attack category)
if 'label' in df.columns:
    print("\n--- Binary Class Distribution (label: 0=Normal, 1=Threat) ---")
    print(df['label'].value_counts(dropna=False))

if 'type' in df.columns:
    print("\n--- Detailed Attack Type Breakdown ---")
    print(df['type'].value_counts(dropna=False))

In [ ]:
# Set visual style for aesthetic security analytics
sns.set_theme(style="darkgrid", palette="mako")
plt.rcParams.update({'figure.titlesize': 14, 'axes.labelsize': 12, 'axes.titlesize': 13})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Normal vs. Suspicious (Threat) Activity Frequency
label_names = {0: 'Normal (0)', 1: 'Suspicious / Attack (1)'}
if 'label' in df.columns:
    plot_labels = df['label'].map(label_names) if df['label'].dtype in [np.int64, int] else df['label']
    sns.countplot(x=plot_labels, ax=axes[0], palette=["#2ec4b6", "#e71d36"])
    axes[0].set_title("Traffic Distribution: Normal vs. Suspicious Activities", fontweight="bold")
    axes[0].set_xlabel("Traffic Category")
    axes[0].set_ylabel("Flow Count")
    
    # Add value count labels on top of bars
    for p in axes[0].patches:
        axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

# Plot 2: Distribution of a Key Network Feature (e.g. Duration or Source Bytes)
feature_to_plot = 'duration' if 'duration' in df.columns else ('src_bytes' if 'src_bytes' in df.columns else df.columns[4])
# Convert to numeric if needed, coerce errors
temp_feature = pd.to_numeric(df[feature_to_plot], errors='coerce').fillna(0)
# Log-transform for clear distribution visualization of skewed network telemetry
log_feature = np.log1p(temp_feature[temp_feature >= 0])

sns.histplot(data=df, x=log_feature, hue='label', kde=True, bins=35, ax=axes[1], palette=["#2ec4b6", "#e71d36"], element="step")
axes[1].set_title(f"Distribution of Network Feature: log1p({feature_to_plot})", fontweight="bold")
axes[1].set_xlabel(f"log(1 + {feature_to_plot})")
axes[1].set_ylabel("Density / Frequency")

plt.tight_layout()
plt.show()

## Task 3: Machine Learning-Based Threat Classification (Decision Tree)
We clean missing data, drop non-generalizing identity columns (IPs, timestamps), encode network protocol categoricals, scale continuous values, train a `DecisionTreeClassifier`, and calculate standard evaluation metrics.

In [ ]:
# Preprocessing: Replace '-' or '?' Zeek null representations with NaN
df_clean = df.replace(['-', '?', ' -', '- '], np.nan)

# Identify target label
target_col = 'label'
y = df_clean[target_col].astype(int)

# Columns to drop: high-cardinality metadata (IP addresses, timestamps) or target leaks ('type')
cols_to_drop = [c for c in ['ts', 'src_ip', 'dst_ip', 'type', 'label'] if c in df_clean.columns]
X_raw = df_clean.drop(columns=cols_to_drop)

# Separate categorical and numerical features
cat_cols = X_raw.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()

# Fill missing values: median for numerical, mode for categorical
for c in num_cols:
    X_raw[c] = pd.to_numeric(X_raw[c], errors='coerce').fillna(X_raw[c].median() if not X_raw[c].dropna().empty else 0)
for c in cat_cols:
    X_raw[c] = X_raw[c].fillna('unknown')

# Encode categorical features using LabelEncoder
for c in cat_cols:
    le = LabelEncoder()
    X_raw[c] = le.fit_transform(X_raw[c].astype(str))

# Normalize numerical features with StandardScaler
scaler = StandardScaler()
X_scaled = X_raw.copy()
X_scaled[num_cols] = scaler.fit_transform(X_scaled[num_cols])

print(f"Feature matrix prepared with shape: {X_scaled.shape}")

In [ ]:
# Stratified train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} samples | Testing set: {X_test.shape[0]} samples")

# Train Decision Tree Classifier
dt_start_time = time.time()
dt_model = DecisionTreeClassifier(max_depth=12, random_state=42)
dt_model.fit(X_train, y_train)
dt_train_time = time.time() - dt_start_time

# Generate Predictions
dt_preds = dt_model.predict(X_test)

# Compute Evaluation Metrics
dt_acc = accuracy_score(y_test, dt_preds)
dt_prec = precision_score(y_test, dt_preds, zero_division=0)
dt_rec = recall_score(y_test, dt_preds, zero_division=0)
dt_f1 = f1_score(y_test, dt_preds, zero_division=0)

print("="*55)
print("       DECISION TREE CLASSIFIER EVALUATION")
print("="*55)
print(f"Training Time : {dt_train_time:.4f} seconds")
print(f"Accuracy      : {dt_acc * 100:.2f}%")
print(f"Precision     : {dt_prec:.4f}")
print(f"Recall        : {dt_rec:.4f}")
print(f"F1-Score      : {dt_f1:.4f}")
print("="*55)
print("\nDetailed Classification Report:")
print(classification_report(y_test, dt_preds, target_names=['Normal', 'Suspicious']))

## Task 4: Deep Learning Threat Classification (MLP Neural Network)
We implement a Multi-Layer Perceptron (`MLPClassifier`) architecture, record exact training time, evaluate using identical metrics, and produce a comparative assessment.

In [ ]:
# Initialize MLP Classifier (Artificial Neural Network)
# Architecture: 2 Hidden Layers (64 nodes, 32 nodes), ReLU activation, Adam optimizer
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=25,
    random_state=42,
    early_stopping=True
)

# Measure and print training time
mlp_start_time = time.time()
print("Training Multi-Layer Perceptron (MLP) Neural Network...")
mlp_model.fit(X_train, y_train)
mlp_train_time = time.time() - mlp_start_time

# Evaluate Deep Learning model
mlp_preds = mlp_model.predict(X_test)
mlp_acc = accuracy_score(y_test, mlp_preds)
mlp_prec = precision_score(y_test, mlp_preds, zero_division=0)
mlp_rec = recall_score(y_test, mlp_preds, zero_division=0)
mlp_f1 = f1_score(y_test, mlp_preds, zero_division=0)

print("="*55)
print("     MLP NEURAL NETWORK (DEEP LEARNING) EVALUATION")
print("="*55)
print(f"Training Time : {mlp_train_time:.4f} seconds")
print(f"Accuracy      : {mlp_acc * 100:.2f}%")
print(f"Precision     : {mlp_prec:.4f}")
print(f"Recall        : {mlp_rec:.4f}")
print(f"F1-Score      : {mlp_f1:.4f}")
print("="*55)
print("\nDetailed Classification Report:")
print(classification_report(y_test, mlp_preds, target_names=['Normal', 'Suspicious']))

In [ ]:
# Model Performance & Architecture Comparison
comparison_df = pd.DataFrame({
    'Metric': ['Training Time (s)', 'Accuracy (%)', 'Precision', 'Recall', 'F1-Score'],
    'Decision Tree (ML)': [
        f"{dt_train_time:.3f}s", f"{dt_acc*100:.2f}%", f"{dt_prec:.4f}", f"{dt_rec:.4f}", f"{dt_f1:.4f}"
    ],
    'MLP Neural Net (DL)': [
        f"{mlp_train_time:.3f}s", f"{mlp_acc*100:.2f}%", f"{mlp_prec:.4f}", f"{mlp_rec:.4f}", f"{mlp_f1:.4f}"
    ]
})

print("\n================ ML vs. DL PERFORMANCE COMPARISON ================")
display(comparison_df)

print(f"""
Comparative Synthesis & Analytical Takeaways:
1. Computational Overhead: The Decision Tree trained in {dt_train_time:.3f}s, whereas MLP required {mlp_train_time:.3f}s (a {mlp_train_time/max(dt_train_time, 1e-4):.1f}x difference). For line-rate edge IDS devices, tree models offer significantly lower training and inference latency.
2. Non-Linear Feature Modeling: Neural Networks excel when learning non-linear, latent cross-protocol features without requiring manual feature engineering, while Decision Trees provide immediate human interpretability (rule extraction for SOC analysts).
3. F1-Score & Robustness: Decision Tree achieved F1={dt_f1:.4f} and MLP achieved F1={mlp_f1:.4f}, demonstrating that both architectures provide strong discriminative capabilities on TON_IoT telemetry.
""")

## Task 5: Ethical Reflection, Aggressive Posture Simulation & False Positives
In operational Security Operations Centers (SOCs), teams often set an **aggressive security threshold** (e.g., classifying any flow with >30% threat probability as malicious) to minimize missed attacks (False Negatives).  
However, this comes at the cost of **False Positives**—blocking legitimate users, causing denial of service, lost productivity, and alert fatigue.

In [ ]:
# Extract continuous threat probabilities from the MLP model
mlp_probs = mlp_model.predict_proba(X_test)[:, 1]

# Apply strict security posture threshold (30% instead of standard 50%)
strict_threshold = 0.30
strict_preds = (mlp_probs >= strict_threshold).astype(int)

# Compute confusion matrix under aggressive posture
cm_strict = confusion_matrix(y_test, strict_preds)
tn, fp, fn, tp = cm_strict.ravel()

total_normal = tn + fp
fp_percentage = (fp / total_normal) * 100

print("="*65)
print(f" AGGRESSIVE SECURITY POSTURE SIMULATION (Threshold > {strict_threshold*100:.0f}%)")
print("="*65)
print(f"True Negatives (Legitimate Allowed)       : {tn:,}")
print(f"False Positives (Legitimate Blocked)      : {fp:,}")
print(f"False Negatives (Attacks Missed)          : {fn:,}")
print(f"True Positives (Attacks Blocked)          : {tp:,}")
print("-"*65)
print(f"Total Legitimate User Sessions Tested     : {total_normal:,}")
print(f"Exact False Positives (Users Blocked)     : {fp:,}")
print(f"False Positive Rate (Legitimate Disrupted): {fp_percentage:.2f}%")
print(f"Attack Recall (Attacks Successfully Caught): {(tp / (tp + fn)) * 100:.2f}%")
print("="*65)

In [ ]:
# Plot Confusion Matrices Comparison: Default 50% vs. Aggressive 30% Threshold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Default threshold confusion matrix
cm_default = confusion_matrix(y_test, mlp_preds)
ConfusionMatrixDisplay(confusion_matrix=cm_default, display_labels=['Normal', 'Suspicious']).plot(
    ax=axes[0], cmap='Blues', colorbar=False
)
axes[0].set_title("Standard Security Posture (Threshold = 0.50)", fontweight='bold')
axes[0].grid(False)

# Aggressive threshold confusion matrix
ConfusionMatrixDisplay(confusion_matrix=cm_strict, display_labels=['Normal', 'Suspicious']).plot(
    ax=axes[1], cmap='Reds', colorbar=False
)
axes[1].set_title(f"Aggressive Security Posture (Threshold = {strict_threshold:.2f})", fontweight='bold')
axes[1].grid(False)

plt.tight_layout()
plt.show()

### Ethical Discussion & Real-World Cyber Dilemma
1. **Denial of Service Against Legitimate Users:** Lowering detection threshold from 0.50 to 0.30 aggressively suppresses False Negatives, ensuring fewer attacks slip through. However, as demonstrated by the simulation, it causes **False Positives** where bona fide user sessions are terminated, leading to accidental self-inflicted Denial of Service.
2. **Operational Alert Fatigue:** When hundreds or thousands of legitimate network transactions are flagged as malicious per hour, Tier-1 Security Operations Center (SOC) analysts suffer from alert fatigue. High false-positive rates frequently lead operators to ignore or disable automated alerts, inadvertently creating worse blind spots.
3. **Regulatory and Business Impact:** In mission-critical sectors (e.g., healthcare IoT, automated manufacturing, financial transactions), disconnecting an operational device due to a False Positive can endanger human safety or disrupt enterprise operations.
4. **Mitigation Recommendation:** Deploy a tiered response architecture—use high-confidence thresholds (>80%) for automated packet dropping, and mid-range probabilities (30% - 79%) for non-disruptive actions (e.g., enhanced telemetry capture, session rate-limiting, step-up multi-factor authentication, and human-in-the-loop analyst triage).